# `doubao-asr` 全接口调用演示 (Demo Notebook)

本 Notebook 演示 **零第三方依赖、高性能豆包输入法语音识别 + 官方 LLM 智能整理 SDK (`doubao-asr`)** 的全部调用方式：

1. **环境检查与零依赖验证**
2. **方式一：一行代码极简同步转写 + 官方 LLM 智能整理 (`transcribe`)**
3. **方式二：多格式一键导出（`Text` / `SRT` 字幕 / `WebVTT` 字幕 / `Markdown` / `JSON`）**
4. **方式三：长音视频自适应静音切分 + 多线程并发极速转写 (`DoubaoASR.transcribe(workers=4)`)**
5. **方式四：生成器流式转写 (`transcribe_stream` — Pull Stream 边解码边出字)**
6. **方式五：协议级实时推流会话 (`asr.create_stream` — Push Stream 麦克风/WebSocket 实时音频流)**
7. **方式六：异步流式生成器 (`asr.atranscribe_stream` — 适用于 FastAPI / SSE / WebSocket 服务端)**
8. **方式七：官方 LLM「智能整理」单独调用 (`organize_text`) 与热词增强 (`hotwords`)**
9. **方式八：底层音频与编解码工具直接调用（C 级 `compute_rms`、VAD 静音切分、`OpusFrameEncoder`）**
10. **方式九：CLI 命令行调用演示 (`python -m doubao_asr`)**

In [1]:
import os
import sys

# 将 src 加入模块搜索路径（若已 pip install -e . 则无需此行）
sys.path.insert(0, os.path.abspath("src"))

import doubao_asr
from doubao_asr import (
    DoubaoASR,
    DoubaoStreamSession,
    transcribe,
    transcribe_stream,
    organize_text,
    OpusFrameEncoder,
    encode_pcm_to_opus_frames,
    compute_rms,
    split_pcm_by_silence,
)

print("doubao_asr 版本:", doubao_asr.__version__)
print("公开导出符号:", doubao_asr.__all__)

doubao_asr 版本: 1.0.0
公开导出符号: ['DoubaoASR', 'DoubaoStreamSession', 'transcribe', 'transcribe_stream', 'organize_text', 'Segment', 'StreamEvent', 'TranscriptResult', 'OpusFrameEncoder', 'encode_pcm_to_opus_frames', 'compute_rms', 'split_pcm_by_silence', 'iter_pcm16k_chunks', 'load_pcm16k']


---
## 方式一：一行代码极简同步转写 + 官方 LLM 智能整理 (`transcribe`)

适用场景：快速将单个音频或视频文件（支持 `.wav` / `.mp3` / `.m4a` / `.mp4` / `.m4s` / `.flac` 等任意格式，或内存 `16kHz PCM bytes`）转为文字，并可选 `organize=True` 自动经过豆包官方 LLM「智能整理」。

In [2]:
# 对示例音频进行一行代码转写（同时开启官方 LLM 智能整理）
res = transcribe("examples/asr_example_zh.wav", organize=True)

print("智能整理成稿:", res.text)
print("ASR 原始文本:", res.raw_text)
print("音频时长:", f"{res.duration}s")
print("转写耗时:", f"{res.elapsed_sec}s")
print("分段明细:", res.segments)

智能整理成稿: 欢迎大家来体验达摩院推出的语音识别模型
ASR 原始文本: 欢迎大家来体验达摩院推出的语音识别模型
音频时长: 5.55s
转写耗时: 4.96s
分段明细: [Segment(index=1, start=0.0, end=5.55, text='欢迎大家来体验达摩院推出的语音识别模型', rms=4000.4, confidence=0.0)]


---
## 方式二：多格式一键导出（`SRT` / `VTT` / `Markdown` / `JSON`）

`transcribe()` 返回的 `TranscriptResult` 对象内置多种标准格式转换器，并可通过 `res.save("path.srt")` 直接保存文件。

In [3]:
print("=== [1] SRT 字幕格式 ===")
print(res.to_srt())

print("=== [2] WebVTT 字幕格式 ===")
print(res.to_vtt())

print("=== [3] Markdown 报告格式 ===")
print(res.to_markdown(title="达摩院语音示例转写稿"))

print("=== [4] JSON 结构化格式 ===")
print(res.to_json())

=== [1] SRT 字幕格式 ===
1
00:00:00,000 --> 00:00:05,550
欢迎大家来体验达摩院推出的语音识别模型

=== [2] WebVTT 字幕格式 ===
WEBVTT

00:00:00.000 --> 00:00:05.550
欢迎大家来体验达摩院推出的语音识别模型

=== [3] Markdown 报告格式 ===
# 达摩院语音示例转写稿

- **音频来源**: `examples/asr_example_zh.wav`
- **音频时长**: `5.5s`
- **转写耗时**: `4.96s`
- **实时倍率**: `1.1x`
- **有效片段**: `1 / 1` 段

## 一、 智能整理成稿

欢迎大家来体验达摩院推出的语音识别模型

## 二、 ASR 原始识别文本

欢迎大家来体验达摩院推出的语音识别模型

## 三、 分段时间轴明细

| 序号 | 时间轴 | 识别文本 |
| :---: | :---: | :--- |
| 01 | `00:00.0 - 00:05.5` | 欢迎大家来体验达摩院推出的语音识别模型 |

=== [4] JSON 结构化格式 ===
{
  "source": "examples/asr_example_zh.wav",
  "duration": 5.55,
  "elapsed_sec": 4.96,
  "text": "欢迎大家来体验达摩院推出的语音识别模型",
  "raw_text": "欢迎大家来体验达摩院推出的语音识别模型",
  "organized_text": "欢迎大家来体验达摩院推出的语音识别模型",
  "segments": [
    {
      "index": 1,
      "start": 0.0,
      "end": 5.55,
      "text": "欢迎大家来体验达摩院推出的语音识别模型",
      "rms": 4000.4,
      "confidence": 0.0
    }
  ]
}


---
## 方式三：长音频自适应 VAD 静音切分 + 多线程并发转写 (`DoubaoASR.transcribe`)

适用场景：几分钟至几小时的 Vlog、播客、网课录音或视频文件。
- 自动在 `[min_segment_sec, max_segment_sec]` 窗口内寻找能量最低的呼吸停顿点切分；
- 默认启用 `workers=4` 多线程并发识别，底层 `WSSConnectionPool` 利用 `x-custom-keepalive: true` 建连仅发一次 `StartTask`，跨分片无限复用 `StartSession`；
- 支持 `organize=True` 对全文做按句切块并发 LLM 智能整理。

In [4]:
# 将两段示例音频拼接构造一段约 16 秒的多句长音频流进行并发转写 + 智能整理演示
with DoubaoASR() as asr:
    pcm1 = asr.load_pcm16k("examples/asr_example_zh.wav")
    pcm2 = asr.load_pcm16k("examples/asr_example.wav")
    silence_1s = b"\x00\x00" * 16000
    long_pcm = pcm1 + silence_1s + pcm2 + silence_1s + pcm1

    long_res = asr.transcribe(
        long_pcm,
        min_segment_sec=4.0,
        max_segment_sec=7.0,
        workers=4,
        organize=True,
        on_segment=lambda seg: print(f"  -> [实时回调 #{seg.index:02d}] [{seg.timestamp}] (RMS={seg.rms}, conf={seg.confidence}) {seg.text}"),
    )

print("\n并发转写完成！总时长:", f"{long_res.duration}s", "| 实际耗时:", f"{long_res.elapsed_sec}s")
print("ASR 原始文本:", long_res.raw_text)
print("智能整理成稿:", long_res.text)

  -> [实时回调 #02] [00:05.6 - 00:09.6] (RMS=349.6, conf=0.9913) 每一天都要快乐哦！
  -> [实时回调 #01] [00:00.0 - 00:05.6] (RMS=2975.4, conf=0.967) 欢迎大家来体验达摩院推出的语音识别模型。
  -> [实时回调 #03] [00:09.6 - 00:15.5] (RMS=2768.1, conf=0.9778) 欢迎大家来体验达摩院推出的语音识别模型。

并发转写完成！总时长: 15.53s | 实际耗时: 2.36s
ASR 原始文本: 欢迎大家来体验达摩院推出的语音识别模型。每一天都要快乐哦！欢迎大家来体验达摩院推出的语音识别模型。
智能整理成稿: 欢迎大家来体验达摩院推出的语音识别模型。每一天都要快乐哦！

欢迎大家来体验达摩院推出的语音识别模型。


---
## 方式四：生成器流式转写接口 (`transcribe_stream` — Pull Stream)

适用场景：希望边读取文件（或边通过 `ffmpeg` 管道解码视频），边向豆包 ASR 服务端流式发包，实时获取**增量中间结果 (`type="partial"`)** 与 **句末标点定稿结果 (`type="final"`)**。

In [5]:
for ev in transcribe_stream("examples/asr_example_zh.wav", chunk_sec=0.8, include_partial=True):
    status = "FINAL  (定稿)" if ev.is_final else "PARTIAL(流式)"
    print(f"[{status}] seg=#{ev.segment_index} seq={ev.seq:02d} [{ev.timestamp}] text={ev.text!r} | delta={ev.delta!r}")

[FINAL  (定稿)] seg=#1 seq=08 [00:00.0 - 00:05.5] text='欢迎大家来体验达摩院推出的语音识别模型' | delta='欢迎大家来体验达摩院推出的语音识别模型'


---
## 方式五：协议级实时推流会话 (`asr.create_stream` — Push Stream)

适用场景：麦克风实时录音（如 `PyAudio` / `sounddevice`）、WebSocket 实时语音流、RTC 音频流。
- 支持以任意帧大小（如每 100ms / 200ms）不断调用 `stream.feed(pcm_chunk)` 推入 `16kHz / 16-bit / 单声道` PCM 数据；
- 检测到说话停顿（VAD）或达到单句时长上限时自动定稿断句，并在同一条 WebSocket 长连接上无缝开启下一句 `StartSession`；
- 结束时通过 `stream.result` 可直接拿到完整的 `TranscriptResult` 并导出字幕。

In [6]:
with DoubaoASR() as asr:
    raw_pcm = asr.load_pcm16k("examples/asr_example.wav")

    # 模拟麦克风每 200ms (6400 字节) 产生一块实时 PCM 数据并推入流式会话
    step_bytes = 6400  # 200ms @ 16kHz 16-bit mono
    with asr.create_stream(chunk_sec=0.6, source_name="mic_simulation") as stream:
        for offset in range(0, len(raw_pcm), step_bytes):
            mic_chunk = raw_pcm[offset : offset + step_bytes]
            for ev in stream.feed(mic_chunk):
                print(f"[Push Event: {ev.type.upper():7s}] [{ev.timestamp}] {ev.text} (delta={ev.delta!r})")

    # 退出 with 块后自动 flush 定稿，直接获取结构化结果
    stream_res = stream.result
    print("\n推流会话最终文本:", stream_res.text)


推流会话最终文本: 每一天都要快乐哦


---
## 方式六：异步流式生成器 (`asr.atranscribe_stream` — AsyncIO)

适用场景：在 `FastAPI`、`Starlette`、`aiohttp`、WebSocket 或 Server-Sent Events (SSE) 异步服务中非阻塞地流式返回识别事件。

In [7]:
import asyncio

async def run_async_demo():
    with DoubaoASR() as asr:
        async for ev in asr.atranscribe_stream("examples/asr_example.wav", chunk_sec=0.8):
            print("Async Event:", ev.to_dict())

asyncio.run(run_async_demo())

Async Event: {'type': 'final', 'segment_index': 1, 'seq': 5, 'start': 0.0, 'end': 2.43, 'text': '每一天都要快乐哦！', 'delta': '每一天都要快乐哦！', 'full_text': '每一天都要快乐哦！', 'is_final': True}


---
## 方式七：官方 LLM「智能整理」单独调用 (`organize_text`)

适用场景：已有口语文本或其它渠道的 ASR 原文，直接调用免鉴权的豆包输入法官方 LLM「智能整理」接口做标点、断句、数字格式化与分段规整。

In [8]:
raw_spoken = "今天天气真不错我们下午三点半去开个会讨论一下deepseek和claude code的集成方案吧另外还要看一下十万行代码的重构进度"
organized = organize_text(raw_spoken)
print("整理前:", raw_spoken)
print("整理后:", organized)

整理前: 今天天气真不错我们下午三点半去开个会讨论一下deepseek和claude code的集成方案吧另外还要看一下十万行代码的重构进度
整理后: 今天天气真不错。我们下午三点半去开个会，讨论一下 deepseek 和 claude code 的集成方案吧。另外，还要看一下十万行代码的重构进度。


---
## 方式八：底层音频与编解码工具直接调用

如果需要单独使用本项目的 **C 级高速 RMS 能量计算**、**自适应静音切分器 (`split_pcm_by_silence`)** 或 **直接编码裸 Opus 帧 (`encode_pcm_to_opus_frames` / `OpusFrameEncoder`)**，可直接从顶层包导入：

In [9]:
with DoubaoASR() as asr:
    pcm_data = asr.load_pcm16k("examples/asr_example_zh.wav")

# 1. C 级 audioop.rms 能量计算
rms_val = compute_rms(pcm_data)
print(f"整段音频 RMS 能量: {rms_val:.1f}")

# 2. 编码为 10ms 裸 Opus 帧列表
opus_frames = encode_pcm_to_opus_frames(pcm_data, sr=16000, frame_ms=10)
print(f"原始 PCM 大小: {len(pcm_data)} 字节 -> 10ms Opus 帧数: {len(opus_frames)} 帧 (总计 {sum(len(f) for f in opus_frames)} 字节)")

# 3. 静音切分演示
segs = split_pcm_by_silence(long_pcm, min_segment_sec=4.0, max_segment_sec=7.0)
for idx, (s_sec, e_sec, sub_pcm, seg_rms) in enumerate(segs, 1):
    print(f"切分片段 #{idx}: {s_sec:.1f}s ~ {e_sec:.1f}s (时长 {e_sec-s_sec:.1f}s, 平均 RMS={seg_rms:.1f})")

整段音频 RMS 能量: 4000.4
原始 PCM 大小: 177494 字节 -> 10ms Opus 帧数: 555 帧 (总计 14145 字节)
切分片段 #1: 0.0s ~ 5.6s (时长 5.6s, 平均 RMS=2975.4)
切分片段 #2: 5.6s ~ 9.6s (时长 4.0s, 平均 RMS=349.6)
切分片段 #3: 9.6s ~ 15.5s (时长 5.9s, 平均 RMS=2768.1)


---
## 方式九：CLI 命令行调用方式 (`python -m doubao_asr`)

除了 Python API，还可以直接在终端使用命令行工具完成批量并发转写、LLM 智能整理或实时流式出字：

In [10]:
import subprocess

cmd = [sys.executable, "-m", "doubao_asr", "examples/asr_example.wav", "-f", "srt"]
env = dict(os.environ, PYTHONPATH=os.path.abspath("src"), PYTHONIOENCODING="utf-8")
proc = subprocess.run(cmd, stdin=subprocess.DEVNULL, capture_output=True, text=True, encoding="utf-8", env=env)
print(proc.stdout)

[01] [00:00.0 - 00:02.4] 每一天都要快乐哦！
1
00:00:00,000 --> 00:00:02,430
每一天都要快乐哦！


